In [1]:
!pip install -q google-generativeai

In [7]:
import google.generativeai as genai
from getpass import getpass
import json

api_key = getpass("Вставьте ваш GOOGLE_API_KEY: ")
genai.configure(api_key=api_key)
model = genai.GenerativeModel('gemini-3.6-flash')

corpus = {
    "sentence": {
        "en": "The bank raised interest rates by two percentage points last quarter.",
        "ru": "Банк повысил процентные ставки на два процентных пункта в прошлом квартале.",
        "kk": "Банк өткен тоқсанда пайыздық мөлшерлемені екі пайыздық тармаққа көтерді."
    },
    "complaint": {
        "en": "Good afternoon. I opened a deposit at your branch in March and was told the rate was fixed for twelve months. In August the rate on my account dropped without any notice. I have attached the contract and the statement. Please explain on what basis the rate was changed and restore the original terms.",
        "ru": "Добрый день. Я открыл депозит в вашем отделении в марте, и мне сказали, что ставка зафиксирована на двенадцать месяцев. В августе ставка по моему счёту снизилась без какого-либо уведомления. Прилагаю договор и выписку. Прошу объяснить, на каком основании была изменена ставка, и восстановить первоначальные условия.",
        "kk": "Қайырлы күн. Мен наурыз айында сіздің бөлімшеңізде депозит аштым, маған мөлшерлеме он екі айға бекітілген деп айтылды. Тамыз айында менің шотымдағы мөлшерлеме ешқандай хабарламасыз төмендеді. Шартты және үзінді көшірмені қоса тіркеп отырмын. Мөлшерлеме қандай негізде өзгертілгенін түсіндіріп, бастапқы шарттарды қалпына келтіруіңізді сұраймын."
    },
    "system_prompt": {
        "en": "You are a support assistant for a retail bank. Answer only from the documents provided. If the answer is not in them, say so. Never invent an account number, a rate or a date.",
        "ru": "Вы — ассистент поддержки розничного банка. Отвечайте только по предоставленным документам. Если ответа в них нет, так и скажите. Никогда не выдумывайте номер счёта, ставку или дату.",
        "kk": "Сіз — бөлшек банктің қолдау көрсету ассистентісіз. Тек берілген құжаттар бойынша жауап беріңіз. Егер жауап оларда болмаса, солай деп айтыңыз. Шот нөмірін, мөлшерлемені немесе күнді ешқашан ойдан шығармаңыз."
    }
}

languages = ["en", "ru", "kk"]

# Указываем None для one_request_billed, как и ожидает Part 3 при отсутствии генерации
results = {"request_tokens": {}, "one_request_billed": None}

print(f"{'Язык':<5} | {'Входные токены':<15}")
print("-" * 25)

for lang in languages:
    combined_prompt = f"{corpus['system_prompt'][lang]}\n\n{corpus['complaint'][lang]}"

    # Считаем только входные токены
    input_tokens = model.count_tokens(combined_prompt).total_tokens
    results["request_tokens"][lang] = input_tokens

    print(f"{lang:<5} | {input_tokens:<15}")

with open("measurements.json", "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

print("\nГотово! Файл measurements.json сохранен. Можно переходить к расчетам стоимости.")

Вставьте ваш GOOGLE_API_KEY: ··········
Язык  | Входные токены 
-------------------------
en    | 100            
ru    | 130            
kk    | 236            

Готово! Файл measurements.json сохранен. Можно переходить к расчетам стоимости.


In [8]:
import json

# Загружаем наши замеры
with open("measurements.json", "r", encoding="utf-8") as f:
    data = json.load(f)

# Цены из prices.py (USD за 1 млн токенов)
models = {
    "haiku-4.5": {"in": 1.00, "out": 5.00},
    "sonnet-5": {"in": 2.00, "out": 10.00},
    "opus-5": {"in": 5.00, "out": 25.00},
    "fable-5.1": {"in": 10.00, "out": 50.00}
}

languages = ["en", "ru", "kk"]
inputs = data["request_tokens"]

# Используем резервное значение, так как генерация текста была заблокирована API
outputs = {lang: 300 for lang in languages}

# Задаем объем: 2000 запросов в день
requests_per_day = 2000
per_year = requests_per_day * 365

def cost_usd(model_key, in_tok, out_tok):
    m = models[model_key]
    return (in_tok * m["in"] + out_tok * m["out"]) / 1_000_000

print("ONE SUPPORT REQUEST -- tokens, and cost in US cents")
print("-" * 55)
print(f"{'':<14} | {'EN':>10} | {'RU':>10} | {'KK':>10}")
print(f"{'input tokens':<14} | {inputs['en']:>10} | {inputs['ru']:>10} | {inputs['kk']:>10}")
print(f"{'output tokens':<14} | {outputs['en']:>10} | {outputs['ru']:>10} | {outputs['kk']:>10}")
for m in ["haiku-4.5", "sonnet-5", "opus-5", "fable-5.1"]:
    cents = [cost_usd(m, inputs[l], outputs[l]) * 100 for l in languages]
    print(f"{m:<14} | {cents[0]:>10.4f} | {cents[1]:>10.4f} | {cents[2]:>10.4f}")

print(f"\nAT {requests_per_day:,} REQUESTS/DAY -- US dollars per year")
print("-" * 55)
for m in ["haiku-4.5", "sonnet-5", "opus-5", "fable-5.1"]:
    yearly = [cost_usd(m, inputs[l], outputs[l]) * per_year for l in languages]
    print(f"{m:<14} | {yearly[0]:>10,.0f} | {yearly[1]:>10,.0f} | {yearly[2]:>10,.0f}")

print("\nTWO RATIOS THAT ARE NOT THE SAME NUMBER")
print("-" * 55)
print(f"{'input only':<14} | {inputs['en']/inputs['en']:>9.2f}x | {inputs['ru']/inputs['en']:>9.2f}x | {inputs['kk']/inputs['en']:>9.2f}x")

base_bill = cost_usd("opus-5", inputs["en"], outputs["en"])
bills = [cost_usd("opus-5", inputs[l], outputs[l]) / base_bill for l in languages]
print(f"{'total bill':<14} | {bills[0]:>9.2f}x | {bills[1]:>9.2f}x | {bills[2]:>9.2f}x")

ONE SUPPORT REQUEST -- tokens, and cost in US cents
-------------------------------------------------------
               |         EN |         RU |         KK
input tokens   |        100 |        130 |        236
output tokens  |        300 |        300 |        300
haiku-4.5      |     0.1600 |     0.1630 |     0.1736
sonnet-5       |     0.3200 |     0.3260 |     0.3472
opus-5         |     0.8000 |     0.8150 |     0.8680
fable-5.1      |     1.6000 |     1.6300 |     1.7360

AT 2,000 REQUESTS/DAY -- US dollars per year
-------------------------------------------------------
haiku-4.5      |      1,168 |      1,190 |      1,267
sonnet-5       |      2,336 |      2,380 |      2,535
opus-5         |      5,840 |      5,949 |      6,336
fable-5.1      |     11,680 |     11,899 |     12,673

TWO RATIOS THAT ARE NOT THE SAME NUMBER
-------------------------------------------------------
input only     |      1.00x |      1.30x |      2.36x
total bill     |      1.00x |      1.02x |   